# ENN583 - Week 4 Prac - Epipolar Geometry
In this practical, we use feature correspondences from the KITTI dataset to estimate camera motion from two images. Before estimating motion, we first investigate how epipolar geometry can identify and reject incorrect feature matches. You will compare RANSAC and USAC fundamental-matrix estimates, test the filtering with deliberately incorrect correspondences, and visualise an epipolar line.

We then use the Essential matrix to recover the relative rotation and translation direction between consecutive camera frames. The practical introduces the coordinate-frame conventions used by OpenCV and KITTI, checks one frame-to-frame estimate, and accumulates the relative motions into a complete trajectory.

A monocular Essential-matrix estimate does not provide the magnitude of translation. For now, we use the ground-truth trajectory only to supply this scale. Recovering metric scale from stereo depth and reprojection error will be explored in Week 5.

### Setting Things Up
As always, we start by setting up the KITTI dataset object and importing the required modules.

In [ ]:
# This code cell is responsible for importing the `kitti_utils` module, which provides utilities for working with the KITTI dataset.
# It first attempts to find the repository root by looking for the presence of the `kitti_utils.py` file in the current directory and its parent directories. If it finds the file, it adds the `support` directory to the Python path so that the module can be imported. 
# If it cannot find the file, it raises a RuntimeError.
from pathlib import Path
import sys

support_dir = (Path.cwd().resolve().parents[1] / "support")
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))
    
import kitti_utils as kitti

In [ ]:
# Load the KITTI sequence using the course support module.
data = kitti.load_kitti_dataset("2011_09_26_drive_0035")

import cv2
from matplotlib import pyplot as plt
import numpy as np

from spatialmath import *
from spatialmath.base import *
from spatialmath.base import sym
from spatialgeometry import *


### Convenience Functions 
Here we define convenience functions for feature extraction and matching. Hint: If you want, you can implement different feature detectors / descriptors or matching algorithms here.

In [ ]:
def detect_features(img, method='sift'):
    """Detect features in an image."""
    
    if method == 'sift':
        # detect features
        sift = cv2.SIFT_create()
        kp, des = sift.detectAndCompute(img, None)
    elif method == 'orb':
        # detect features
        orb = cv2.ORB_create()
        kp, des = orb.detectAndCompute(img, None)
    else:
        # not implemented error
        raise NotImplementedError('Unknown feature detection method.')
        
    return kp, des

def match_features(des1, des2):
    """Match features between two images."""
    
    # match features
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    matches = bf.match(des1, des2)
    
    return matches

## Filtering Feature Matches with the Fundamental Matrix

Descriptor matching will usually include some incorrect correspondences. A fundamental matrix describes the epipolar geometry between two camera views, so matches that do not agree with this geometry can be rejected. `cv2.findFundamentalMat()` estimates the matrix and returns a mask identifying the inliers.

The example below compares classic RANSAC with a USAC estimator when the installed OpenCV version provides one.

In [ ]:
from time import perf_counter

def show_correspondences(ax, image1, image2, points1, points2, title='', max_matches=100):
    """Draw corresponding points on two images placed side by side."""
    combined = np.hstack((image1, image2))
    ax.imshow(combined, cmap='gray')

    indices = np.arange(len(points1))

    # Limit the number of lines so that the geometry remains visible.
    if len(indices) > max_matches:
        indices = np.random.default_rng(0).choice(indices, max_matches, replace=False)

    offset = image1.shape[1]
    for index in indices:
        colour = plt.cm.hsv(index / max(len(points1), 1))
        x1, y1 = points1[index]
        x2, y2 = points2[index]
        ax.plot([x1, x2 + offset], [y1, y2], color=colour, linewidth=0.7)
        ax.scatter([x1, x2 + offset], [y1, y2], color=[colour], s=5)

    ax.set_title(f'{title} ({len(indices)} shown)')
    ax.axis('off')


# Load two images. Use the left images [0] from the stereo pair at frames 10 and 20, and convert them to grayscale.
image1 = cv2.cvtColor(data.stereo(10)[0], cv2.COLOR_RGB2GRAY)
image2 = cv2.cvtColor(data.stereo(15)[0], cv2.COLOR_RGB2GRAY)

# detect the keypoints and descriptors in both images using our helper function. 
# You can change the method to 'orb' to use ORB instead of SIFT. Or you can implement your own feature detection method and use it here.
keypoints1, descriptors1 = detect_features(image1, method='sift')
keypoints2, descriptors2 = detect_features(image2, method='sift')

# match features using the helper
feature_matches = match_features(descriptors1, descriptors2)

# convert the keypoints to numpy arrays of shape (N, 2) for use with OpenCV's findFundamentalMat function.
points1 = np.float32([keypoints1[match.queryIdx].pt for match in feature_matches])
points2 = np.float32([keypoints2[match.trainIdx].pt for match in feature_matches])

# visualise the matches using the helper function. 
plt.figure(figsize=(16, 8))
show_correspondences(plt.gca(), image1, image2, points1, points2, title='Candidate matches before RANSAC filtering')
plt.show()


# Use RANSAC to find the fundamental matrix and the inlier mask. The inlier mask is a binary array indicating which matches are considered inliers by RANSAC.
# Inspect the other parameters: 
# ransacReprojThreshold is the maximum allowed reprojection error in pixels to treat a point pair as an inlier, and confidence is the probability that the estimated matrix is correct.
start_time = perf_counter()
F_ransac, mask_ransac = cv2.findFundamentalMat(points1, points2, method=cv2.FM_RANSAC, ransacReprojThreshold=1.0, confidence=0.999)
ransac_time = perf_counter() - start_time
ransac_inliers = mask_ransac.ravel().astype(bool)
ransac_points1 = points1[ransac_inliers]
ransac_points2 = points2[ransac_inliers]

# Visualise the matches retained by RANSAC using the helper function. 
plt.figure(figsize=(16, 8))
show_correspondences(plt.gca(), image1, image2, ransac_points1, ransac_points2, title='Matches retained by RANSAC')
plt.show()


# USAC MAGSAC is available in recent OpenCV versions. It is a more advanced method than the standard RANSAC, and can provide better results in some cases. 
# If it is available, we will use it to find the fundamental matrix and the inlier mask, and visualise the matches retained by USAC MAGSAC.
usac_available = hasattr(cv2, 'USAC_MAGSAC')
if usac_available:
    start_time = perf_counter()
    F_usac, mask_usac = cv2.findFundamentalMat(
        points1, points2, method=cv2.USAC_MAGSAC, ransacReprojThreshold=1.0, confidence=0.999
    )
    usac_time = perf_counter() - start_time
    usac_inliers = mask_usac.ravel().astype(bool)
    usac_points1 = points1[usac_inliers]
    usac_points2 = points2[usac_inliers]

    plt.figure(figsize=(16, 8))
    show_correspondences(plt.gca(), image1, image2, usac_points1, usac_points2, title='Matches retained by USAC MAGSAC')
    plt.show()
else:
    print('USAC MAGSAC is not available in this OpenCV build.')

# Print timing and filtering statistics for the two estimators.
number_of_matches = len(feature_matches)
number_of_ransac_inliers = np.count_nonzero(ransac_inliers)
print(f'Candidate matches: {number_of_matches}')
print(f'RANSAC: {number_of_ransac_inliers} inliers ({100 * number_of_ransac_inliers / number_of_matches:.1f}%), ' 
      f'{number_of_matches - number_of_ransac_inliers} rejected, {1000 * ransac_time:.2f} ms')

if usac_available:
    number_of_usac_inliers = np.count_nonzero(usac_inliers)
    estimator_disagreements = np.count_nonzero(ransac_inliers != usac_inliers)
    print(f'USAC MAGSAC: {number_of_usac_inliers} inliers ({100 * number_of_usac_inliers / number_of_matches:.1f}%), ' 
          f'{number_of_matches - number_of_usac_inliers} rejected, {1000 * usac_time:.2f} ms')
    print(f'RANSAC and USAC disagree on {estimator_disagreements} / {number_of_matches} matches.')



### Experiment: Adding Deliberately Incorrect Matches

To make the filtering behaviour clearer, we add random point pairs to the genuine feature correspondences. Each pair is sampled independently in the two images, so it is very unlikely to satisfy the same epipolar geometry.

In [ ]:
# Generate reproducible random correspondences independently in each image.
rng = np.random.default_rng(4)
number_of_random_matches = 100
height, width = image1.shape
random_points1 = rng.uniform([0, 0], [width, height], (number_of_random_matches, 2)).astype(np.float32)
random_points2 = rng.uniform([0, 0], [width, height], (number_of_random_matches, 2)).astype(np.float32)

# Add the random correspondences to the genuine feature matches from the previous cell.
points1_with_outliers = np.vstack((points1, random_points1))
points2_with_outliers = np.vstack((points2, random_points2))

# call RANSAC with the augmented set of correspondences
start_time = perf_counter()
F_ransac_with_outliers, mask_ransac_with_outliers = cv2.findFundamentalMat(points1_with_outliers, points2_with_outliers,method=cv2.FM_RANSAC, ransacReprojThreshold=1.0, confidence=0.999)
ransac_with_outliers_time = perf_counter() - start_time

ransac_with_outliers = mask_ransac_with_outliers.ravel().astype(bool)
genuine_inliers = ransac_with_outliers[:len(points1)]
random_inliers = ransac_with_outliers[len(points1):]

# First show all deliberately injected correspondences.
plt.figure(figsize=(16, 8))
show_correspondences(
    plt.gca(), image1, image2, random_points1, random_points2,
    title='Injected random matches', max_matches=number_of_random_matches
)
plt.show()

# Then show only the injected correspondences that RANSAC classified as inliers.
plt.figure(figsize=(16, 8))
show_correspondences(
    plt.gca(), image1, image2, random_points1[random_inliers], random_points2[random_inliers],
    title='Random matches retained by RANSAC',
    max_matches=number_of_random_matches
)
plt.show()

# Report genuine and deliberately injected matches separately.
number_of_genuine_inliers = np.count_nonzero(genuine_inliers)
number_of_random_inliers = np.count_nonzero(random_inliers)
number_of_augmented_matches = len(points1_with_outliers)
print(f'Augmented candidate matches: {number_of_augmented_matches} ' 
      f'({len(points1)} genuine + {number_of_random_matches} random)')
print(f'Genuine matches retained: {number_of_genuine_inliers} / {len(points1)} ' 
      f'({100 * number_of_genuine_inliers / len(points1):.1f}%)')
print(f'Random matches retained: {number_of_random_inliers} / {number_of_random_matches} ' 
      f'({100 * number_of_random_inliers / number_of_random_matches:.1f}%)')
print(f'RANSAC estimation time: {1000 * ransac_with_outliers_time:.2f} ms')

### Experiment: Visualising an Epipolar Line

Choose a pixel in the first image below. The fundamental matrix maps that pixel to an epipolar line in the second image.

In [ ]:
# Load two images from KITTI. Experiment by changing the frame numbers, or using the left and right images from the stereo pair ([0] and [1]). The images are converted to grayscale for feature detection.
epipolar_image1 = cv2.cvtColor(data.stereo(10)[0], cv2.COLOR_RGB2GRAY)
epipolar_image2 = cv2.cvtColor(data.stereo(12)[0], cv2.COLOR_RGB2GRAY)


# Detect and match features
epipolar_keypoints1, epipolar_descriptors1 = detect_features(epipolar_image1, method='sift')
epipolar_keypoints2, epipolar_descriptors2 = detect_features(epipolar_image2, method='sift')
epipolar_matches = match_features(epipolar_descriptors1, epipolar_descriptors2)

# convert the keypoints to numpy arrays of shape (N, 2) for use with OpenCV's findFundamentalMat function.
epipolar_points1 = np.float32([epipolar_keypoints1[match.queryIdx].pt for match in epipolar_matches])
epipolar_points2 = np.float32([epipolar_keypoints2[match.trainIdx].pt for match in epipolar_matches])

# Estimate the fundamental matrix from feature matches.
method = cv2.USAC_MAGSAC if hasattr(cv2, 'USAC_MAGSAC') else cv2.FM_RANSAC
F, _ = cv2.findFundamentalMat(epipolar_points1, epipolar_points2, method=method, ransacReprojThreshold=1.0, confidence=0.999)

# Choose a pixel [x, y] in the first image. Experiment with different pixels here and see how the corresponding epipolar line changes in the second image. 
# The pixel coordinates are in (x, y) format, where x is the column index and y is the row index.
pixel = np.float32([200, 200])

# OpenCV has a helper function to compute corresponding epipolar lines.
# The epipolar line has the form ax + by + c = 0 in the second image.
a, b, c = cv2.computeCorrespondEpilines(pixel.reshape(1, 1, 2), 1, F).reshape(3)

# The epipolar line can be plotted by computing the y-coordinates for two x-coordinates (the left and right edges of the image). 
# The line is defined by the equation ax + by + c = 0, which can be rearranged to solve for y in terms of x: y = -(a * x + c) / b. 
# This gives us the y-coordinates corresponding to the left and right edges of the image, allowing us to draw the epipolar line across the entire width of the second image.
x = np.array([0, epipolar_image2.shape[1] - 1])
y = -(a * x + c) / b

# Visualise the selected pixel in the first image and the corresponding epipolar line in the second image. The pixel is marked with a red dot, and the epipolar line is drawn in red.
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(epipolar_image1, cmap='gray')
axes[0].scatter(pixel[0], pixel[1], color='red', s=50)
axes[0].set_title('Selected pixel in image 1')

axes[1].imshow(epipolar_image2, cmap='gray')
axes[1].plot(x, y, color='red', linewidth=2)
axes[1].set_title('Epipolar line in image 2')

for axis in axes:
    axis.set_xlim(0, epipolar_image1.shape[1] - 1)
    axis.set_ylim(epipolar_image1.shape[0] - 1, 0)
    axis.axis('off')
plt.tight_layout()
plt.show()

## Use the Essential Matrix to Build a 2D-2D Motion Pipeline

Here we use OpenCV's inbuilt functions to estimate the camera motion from 2D-2D correspondences. This example shows how to use the `findEssentialMat` and `recoverPose` functions. 
We expect two input images in grayscale format, along with the camera matrix K and return a SE3 object from the spatial math toolbox.

Remember: Estimating the camera motion from the Essential matrix cannot give us the absolute scale of the translation, only the direction. We are going to cheat a little bit here and determine the absolute scale factor from the ground truth motion.

Next week we will explore how to determine the scale factor from stereo information, using optimisation and minimisation of the reprojection error.

The code below will do quite a bit of maths with coordinate transforms. Make sure you understand what the code is doing. Are points being transformed from camera 1 to camera 2, or vice versa? Are they being transformed from the camera frame to the world frame, or vice versa? If you are unsure how the SE3 transforms work, please review the lecture notes from earlier semesters.

Important points:
- OpenCV camera coordinates use $x$ right, $y$ down and $z$ forward.
- KITTI ground-truth coordinates use $x$ forward, $y$ left and $z$ up.
- `recoverPose()` returns $^{c_2}T_{c_1}$, written `T_c2_c1` in code. It maps points from camera 1 into camera 2. We invert it to obtain `T_c1_c2`, the pose of camera 2 relative to camera 1.
- The inlier mask from `findEssentialMat()` is passed into `recoverPose()`, and both functions use the camera calibration matrix.

In [ ]:
def estimate_2D2D_motion(image1, image2, K):
    # Detect and match features.
    keypoints1, descriptors1 = detect_features(image1, method='sift')
    keypoints2, descriptors2 = detect_features(image2, method='sift')
    matches = match_features(descriptors1, descriptors2)

    points1 = np.float32([keypoints1[match.queryIdx].pt for match in matches])
    points2 = np.float32([keypoints2[match.trainIdx].pt for match in matches])

    # Estimate the Essential matrix using RANSAC.
    E, essential_mask = cv2.findEssentialMat(
        points1, points2, cameraMatrix=K, method=cv2.RANSAC,
        prob=0.999, threshold=1.0
    )
    number_of_essential_inliers = np.count_nonzero(essential_mask)

    # Recover the relative camera pose using only the RANSAC inliers.
    _, R, t, pose_mask = cv2.recoverPose(
        E, points1, points2, cameraMatrix=K, mask=essential_mask.copy()
    )
    
    # R and t are the rotation and translation for a SE3 that map points from camera 1 to camera 2. 
    T_c2_c1 = SE3.Rt(R, t.ravel())  # t.ravel() is used to convert the translation vector from a column vector to a 1D array, which is the expected input format for the SE3.Rt constructor.
    
    # The pose of camera 2 relative to camera 1 is the inverse of this transformation.
    T_c1_c2 = T_c2_c1.inv()

    return (
        T_c1_c2,
        len(matches),
        number_of_essential_inliers,
        np.count_nonzero(pose_mask),
    )

### Check One Frame Pair

Before processing the complete sequence, check that one estimate is sensible. The recovered translation has unit length because a monocular Essential matrix provides direction, but not metric scale.

In [ ]:
K = data.camera_calibration(camera=2)['K']
image1 = cv2.cvtColor(data.stereo(0)[0], cv2.COLOR_RGB2GRAY)
image2 = cv2.cvtColor(data.stereo(1)[0], cv2.COLOR_RGB2GRAY)

# Estimate T_c1_c2 and return match-filtering statistics.
T_c1_c2, candidate_count, essential_count, pose_count = estimate_2D2D_motion(image1, image2, K)

# Convert the camera motion into the initial KITTI/IMU frame.
T_cam_imu_matrix = data.camera_calibration(camera=2)['T_cam_imu']
T_cam_imu = SE3.Rt(T_cam_imu_matrix[:3, :3], T_cam_imu_matrix[:3, 3], check=False)

# T_c1_c2 describes the relative motion in camera coordinates, while the KITTI
# ground-truth poses are expressed in IMU coordinates. The camera and IMU are
# rigidly connected by the fixed calibration T_cam_imu.
# Multiplying on both sides changes the coordinate basis of the relative pose
# from camera coordinates to IMU coordinates. Look up “SE(3) change of basis”
# or “conjugating a rigid-body transformation” for more information.
estimated_imu_motion = T_cam_imu.inv() * T_c1_c2 * T_cam_imu
direction_in_gt_axes = estimated_imu_motion.t
direction_in_gt_axes /= np.linalg.norm(direction_in_gt_axes)

# this is the true translation vector between the two frames, expressed in the KITTI/IMU frame. 
# We can use it to compute the ground-truth direction of motion (gt_direction, unit length).
gt_displacement = data.ground_truth_pose(1).t - data.ground_truth_pose(0).t
gt_direction = gt_displacement / np.linalg.norm(gt_displacement)

print(f'Candidate matches: {candidate_count}')
print(f'Essential-matrix inliers: {essential_count}')
print(f'Pose-recovery inliers: {pose_count}')
# We expect the estimated _direction_ to be close to the ground-truth direction, but they may not be exactly equal due to noise and estimation errors.
# For forward motion, the x-component of the direction should be positive, and the y- and z-components should be close to zero.
print('Estimated direction in GT axes:', np.round(direction_in_gt_axes, 3))
print('Ground-truth direction:         ', np.round(gt_direction, 3))

### Estimate the Complete Trajectory

For now, we use ground truth only to obtain the distance travelled between adjacent frames. The estimated rotation and translation direction still come entirely from the images.

Each relative motion is converted from camera coordinates into IMU coordinates immediately. We then accumulate the poses directly in the same initial-IMU frame used by `ground_truth_pose()`.

In [ ]:
# Estimate the frame-to-frame motions.
relative_motions = []
candidate_counts = []
essential_inlier_counts = []
pose_inlier_counts = []

for frame in range(data.frame_count - 1):
    image1 = cv2.cvtColor(data.stereo(frame)[0], cv2.COLOR_RGB2GRAY)
    image2 = cv2.cvtColor(data.stereo(frame + 1)[0], cv2.COLOR_RGB2GRAY)

    relative_motion, candidate_count, essential_count, pose_count = estimate_2D2D_motion(image1, image2, K)
    relative_motions.append(relative_motion)
    candidate_counts.append(candidate_count)
    essential_inlier_counts.append(essential_count)
    pose_inlier_counts.append(pose_count)

    if (frame + 1) % 25 == 0:
        print(f'Processed {frame + 1} frame pairs')

# Load the fixed transform between the KITTI IMU and camera.
T_cam_imu_matrix = data.camera_calibration(camera=2)['T_cam_imu']
T_cam_imu = SE3.Rt(
    T_cam_imu_matrix[:3, :3], T_cam_imu_matrix[:3, 3], check=False
)

# Accumulate the estimated poses directly in the initial IMU frame.
# In the IMU frame, X points forward, Y points left and Z points up.
estimated_imu_trajectory = [SE3()]
gt_scales = []

for frame, T_c1_c2 in enumerate(relative_motions):
    # Use GT only for the distance travelled, not its direction or rotation.
    # We are cheating here to get the scale factor for the translation vector, but in a real application, you would need to estimate this from the stereo images or other sensors.
    position1 = data.ground_truth_pose(frame).t
    position2 = data.ground_truth_pose(frame + 1).t
    scale = np.linalg.norm(position2 - position1)
    gt_scales.append(scale)

    scaled_camera_motion = SE3.Rt(T_c1_c2.R, scale * T_c1_c2.t)

    # Express this frame-to-frame motion in the corresponding IMU frames.
    relative_imu_motion = T_cam_imu.inv() * scaled_camera_motion * T_cam_imu

    # Accumulate imu0Timui directly.
    estimated_imu_trajectory.append(
        estimated_imu_trajectory[-1] * relative_imu_motion
    )

ground_truth_trajectory = [
    data.ground_truth_pose(frame) for frame in range(data.frame_count)
]

# collect the positions of the estimated and ground truth trajectories for plotting
estimated_positions = np.array([pose.t for pose in estimated_imu_trajectory])
ground_truth_positions = np.array([pose.t for pose in ground_truth_trajectory])

# Plot the estimated and ground truth trajectories, the GT scale used per frame, and the number of matches and inliers for each frame pair.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(estimated_positions[:, 0], estimated_positions[:, 1], label='Estimated')
axes[0].plot(ground_truth_positions[:, 0], ground_truth_positions[:, 1], label='Ground truth')
axes[0].set_title('Trajectory')
axes[0].set_xlabel('x [m]')
axes[0].set_ylabel('y [m]')
axes[0].axis('equal')
axes[0].grid()
axes[0].legend()

axes[1].plot(gt_scales)
axes[1].set_title('GT scale used per frame')
axes[1].set_xlabel('Frame pair')
axes[1].set_ylabel('Distance [m]')
axes[1].grid()

axes[2].plot(candidate_counts, label='Candidate matches')
axes[2].plot(essential_inlier_counts, label='Essential inliers')
axes[2].plot(pose_inlier_counts, label='Pose inliers')
axes[2].set_title('Match filtering')
axes[2].set_xlabel('Frame pair')
axes[2].set_ylabel('Number of matches')
axes[2].grid()
axes[2].legend()

plt.tight_layout()
plt.show()

worst_frame = np.argmin(pose_inlier_counts)
print(f'Least reliable frame pair: {worst_frame} -> {worst_frame + 1}')
print(f'Essential-matrix inliers: {essential_inlier_counts[worst_frame]}')
print(f'Pose-recovery inliers: {pose_inlier_counts[worst_frame]}')

### Interpreting Unreliable Motion Estimates

A high Essential-matrix inlier count does not guarantee a reliable translation estimate. If very few points pass the positive-depth test in `recoverPose()`, the images may contain insufficient translational parallax. This can happen when the vehicle is stationary, moving very slowly or undergoing mostly rotation. A practical visual-odometry system should reject or otherwise handle such an estimate instead of accumulating it blindly.

### Reflection Questions

1. Why can rotation be recovered metrically while translation is known only up to scale?
2. Why might `recoverPose()` return very few pose inliers even when the Essential matrix has many inliers?